# 2D → 3D Pipeline — Colab 1-Click Runbook (P6)

Chạy toàn bộ hệ thống trên **Google Colab Free (T4)** và mở Web UI từ máy tính qua **Cloudflare Tunnel**.

Thứ tự: **Runtime ▸ Change runtime type ▸ T4 GPU** → Run all (Cell 1 → Cell 6).

| Cell | Việc |
| :--: | :--- |
| 1 | Clone repo + cài dependencies |
| 2 | Clone TripoSR + tải weights |
| 3 | (Tùy chọn) Upload ảnh test |
| 4 | Bật FastAPI + Cloudflare Tunnel → in ra URL công khai |
| 5 | Smoke test API bằng `curl` |
| 6 | Dừng server |

In [ ]:
# Cell 1: Clone repo + cài dependencies
import os
if not os.path.isdir('Img2d-to-3d'):
    !git clone -q https://github.com/dduy26/Img2d-to-3d.git
%cd /content/Img2d-to-3d

!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx scikit-image opencv-python-headless
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/briaai/RMBG-2.0.git || echo 'RMBG-2.0 dùng transformers AutoModel, bỏ qua bước git'
print('Deps OK')

In [ ]:
# Cell 2: TripoSR (động cơ cứu hộ) — cần cho nhánh Quality FAIL + chế độ 1 ảnh
import os
if not os.path.isdir('/content/TripoSR'):
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
!pip install -q -r /content/TripoSR/requirements.txt
!cp -r /content/TripoSR/tsr notebook/backend/tsr
print('TripoSR source OK (weights ~2GB tự tải ở lần chạy đầu)')

In [ ]:
# Cell 3 (TÙY CHỌN): Upload ảnh test 4–8 góc. Bỏ qua nếu đã có data/input/multi_view.
from google.colab import files
import os, shutil
up = files.upload()  # chọn 4–8 ảnh .jpg/.png
os.makedirs('data/input/multi_view', exist_ok=True)
for name in up:
    shutil.move(name, f'data/input/multi_view/{name}')
print(sorted(os.listdir('data/input/multi_view')))

In [ ]:
# Cell 4: Bật FastAPI + Cloudflare Tunnel (KHÔNG cần tài khoản Cloudflare)
import subprocess, time, os

# cloudflared binary
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Uvicorn chạy nền, cwd = notebook/backend (app.py tạo temp_uploads/ & outputs/ theo cwd)
server = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/Img2d-to-3d/notebook/backend',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# Chờ server sẵn sàng (health check, không sleep mù)
import urllib.request
for i in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            if r.status == 200:
                print('Server READY:', r.read().decode())
                break
    except Exception:
        time.sleep(2)
else:
    print('Server KHÔNG khởi động. Log:')
    print(server.stdout.read()[-3000:])

# Tunnel công khai
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
import re
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print('\n🌐 WEB UI  :', url)
        print('📘 SWAGGER :', url + '/docs')
        break
if url is None:
    print('Không lấy được URL tunnel — xem log trên. Thử lại Cell 4.')

In [ ]:
# Cell 5: Smoke test API (chế độ đa ảnh nếu có >=4 ảnh, ngược lại chế độ 1 ảnh)
import glob, subprocess, json
imgs = sorted(glob.glob('/content/Img2d-to-3d/data/input/multi_view/view_*.jpg')) or \
       sorted(glob.glob('/content/Img2d-to-3d/data/input/multi_view/*.jpg'))
if len(imgs) >= 4:
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/']
    for p in imgs[:6]:
        cmd += ['-F', f'files=@{p}']
else:
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/single/', '-F', f'file=@{imgs[0]}']
out = subprocess.run(cmd, capture_output=True, text=True).stdout
print(json.dumps(json.loads(out), indent=2, ensure_ascii=False) if out else 'Không có phản hồi')

In [ ]:
# Cell 6: Dừng server + tunnel khi xong
tunnel.terminate(); server.terminate()
print('Đã dừng. File .glb nằm ở notebook/backend/outputs/')